# Подготовка FPB + MyFoodRepo v2.1

Объединённый датасет YOLO detection: `class_id x_center y_center width height`, координаты нормализованы. FPB сохраняет исходные рамки и ID, шестое поле массы отбрасывается. MyFoodRepo содержит несколько размеченных объектов на фото: их классы переводятся из разреженных COCO ID в общий словарь, рамки вычисляются по полигонам.

**Food-101 исключён из обучения:** у него нет объектной разметки, а на многих фото несколько блюд. В конце ноутбука можно скачать отдельный сэмпл без классов и рамок для проверки предсказаний. Он не включается в train/val и не подходит для расчёта detection-метрик.

Перед загрузкой выполните ноутбук 01 и проверьте соответствия классов.

In [1]:
from pathlib import Path
import os
import sys
import json
from collections import Counter

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/dataset_pipeline.py").is_file()), None)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(DATA / "cache/matplotlib"))

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from scripts import dataset_pipeline as dp
from scripts import myfoodrepo as mfr

## 1. Настройки

`MYFOODREPO_SOURCE = "archives"` распаковывает уже скачанные train и valid. Пути задаются ниже. Режим `"download"` загружает train с зеркала Kaggle; valid в обоих режимах берётся из локального архива, поскольку в проверенном зеркале его нет. FPB загружается прежним способом.

`preview` и `full` сохраняются отдельно в `data/processed/fpb_myfoodrepo_preview/` и `fpb_myfoodrepo_full/`. При `OVERWRITE=False` готовая сборка читается без повторной загрузки, распаковки, хеширования и объединения. После смены источника или решений включите `OVERWRITE=True` для первой пересборки. Этот флаг пересобирает processed; уже подготовленные исходные файлы переиспользуются.

Для проверки используйте `MODE = "preview"`, затем `"full"`. В full входят 54 392 train и 946 valid MyFoodRepo, а также все опубликованные FPB. При первом импорте архивы распаковываются целиком даже для preview; последующие запуски пропускают завершённую распаковку.


In [ ]:
MODE = "full"           # "preview" — проверочный сэмпл; "full" — все изображения.
OVERWRITE = False          # True — пересобрать результат выбранного режима.
SEED = 42                  # Воспроизводимый выбор сэмпла; авторские train/valid сохраняются.
WORKERS = 8                # Потоки загрузки и проверки изображений.
SAMPLES_PER_CLASS = 2      # Число целевых примеров на класс в preview.
MAX_SCAN = 1600            # Лимит поиска разметки FPB для preview.

MYFOODREPO_SOURCE = "archives"  # "archives" — локальный train; "download" — train с зеркала.
MYFOODREPO_TRAIN_ARCHIVE = DATA / "public_training_set_release_2.1.tar"
MYFOODREPO_VALID_ARCHIVE = DATA / "public_validation_set_release_2.1.tar"  # Нужен в обоих режимах.

DOWNLOAD_FOOD101_SAMPLE = False  # Отдельные фотографии для проверки предсказаний.
FOOD101_SAMPLE_SIZE = 20
OVERWRITE_FOOD101 = False

cats = dp.catalogs(
    DATA, myfoodrepo_source=MYFOODREPO_SOURCE,
    train_archive=MYFOODREPO_TRAIN_ARCHIVE, valid_archive=MYFOODREPO_VALID_ARCHIVE,
)
display(pd.DataFrame([{"source": source, "classes": len(info["names"]), "revision": info["revision"]}
                      for source, info in cats.items()]))

OUTPUT = dp.dataset_path(DATA, MODE)
DECISIONS_PATH = DATA / "review/fpb_myfoodrepo_class_decisions.csv"
if not DECISIONS_PATH.exists():
    raise RuntimeError("Сначала выполните ноутбук 01 и сохраните решения о классах.")
decisions = pd.read_csv(DECISIONS_PATH)
dp.validate_decisions(decisions, cats)
names, source_to_id, mapping_rows = dp.class_mapping(cats, decisions)
display(pd.DataFrame(mapping_rows).query("source == 'myfoodrepo'"))
print("Классов:", len(names), "Режим:", MODE)


,source,classes,revision
0,fpb,138,53fcacf4b9dbe24c1c6ffa5a2cdb9d8c502e482f
1,myfoodrepo,323,0b7ccc6d196b


,source,source_class_id,source_class,class_id,class_name,rule
138,myfoodrepo,50,beetroot-steamed-without-addition-of-salt,138,beetroot-steamed-without-addition-of-salt,myfoodrepo_separate
139,myfoodrepo,100022,green_bean_steamed_without_addition_of_salt,139,green_bean_steamed_without_addition_of_salt,myfoodrepo_separate
140,myfoodrepo,100031,watermelon_fresh,140,watermelon_fresh,myfoodrepo_separate
141,myfoodrepo,100049,tuna_in_oil_drained,141,tuna_in_oil_drained,myfoodrepo_separate
142,myfoodrepo,100057,sweet_potato,142,sweet_potato,myfoodrepo_separate
...,...,...,...,...,...,...
456,myfoodrepo,101355,yogurt,413,yogurt,myfoodrepo_separate
457,myfoodrepo,101358,applesauce,414,applesauce,myfoodrepo_separate
458,myfoodrepo,101360,red_cabbage_raw,415,red_cabbage_raw,myfoodrepo_separate
459,myfoodrepo,101361,cream_double_cream_heavy_cream_45,416,cream_double_cream_heavy_cream_45,myfoodrepo_separate


Классов: 417 Режим: full


## 2. Подготовка источников

FPB: изображения и исходные рамки без массы. MyFoodRepo v2.1: локальные tar/tar.gz либо train из [Kaggle, версия 24](https://www.kaggle.com/datasets/rinatkharisov/detectron2ra/versions/24) и локальный valid. Проверяются 323 одинаковые категории в обоих разбиениях, 54 392 / 946 фото и 100 256 / 1 708 исходных аннотаций. В режиме archives MyFoodRepo не скачивается.

Распаковка показывает tqdm и сохраняет изображения в `data/raw/myfoodrepo_v2_1_archives/{train,val}/images/`. Ссылки и выход за пределы целевой папки в tar запрещены. JSON, сведения об архиве и отметки завершения кешируются в `data/metadata/myfoodrepo_v2_1_archives/{train,val}/`. Прерванную распаковку можно продолжить; завершённая повторно не выполняется.

Рамки рассчитываются по полигонам: вырожденные компоненты игнорируются при наличии валидных, точные повторы объекта удаляются. Аудит сохраняется рядом с JSON соответствующего источника. Проверяются реальные размеры фото; невалидные координаты останавливают импорт. tqdm показывает прогресс загрузки, распаковки и сборки.


In [ ]:
prepared = None if OVERWRITE else dp.load_dataset(
    DATA, cats, decisions, mode=MODE, seed=SEED,
)
records = None
if prepared is None:
    if MODE == "full":
        import shutil
        if shutil.disk_usage(DATA).free < 15 * 2**30:
            raise RuntimeError("Для полной подготовки освободите не менее 15 ГБ или начните с preview.")
        records = dp.prepare_full(DATA, cats, workers=WORKERS)
    else:
        records, coverage = dp.preview_samples(DATA, cats, per_class=SAMPLES_PER_CLASS,
                                               max_scan=MAX_SCAN, workers=WORKERS)
        print("Недостаточное покрытие FPB:", coverage["missing"])
    display(pd.DataFrame(records).groupby(["source", "source_split"]).size().rename("images").to_frame())
for split in cats["myfoodrepo"]["splits"]:
    if split in cats["myfoodrepo"]["archives"]:
        audit_path = DATA / "metadata/myfoodrepo_v2_1_archives" / split / "import_audit.json"
    else:
        audit_path = DATA / "metadata/myfoodrepo_v2_1/import_audit.json"
    if audit_path.exists():
        print("Коррекции MyFoodRepo:", split)
        display(pd.Series(Counter(row["reason"] for row in dp.read_json(audit_path)), name="annotation_corrections"))


FPB: фото и разметка:   0%|          | 0/14076 [00:00<?, ?фото/s]

## 3. Объединение и разбиения

- FPB сохраняет ID 0–137 и исходные train/val. Его test без открытой разметки находится отдельно в `fpb_test_unlabeled`.
- MyFoodRepo сохраняет авторские разбиения: **train → train, valid → val**. Случайное выделение val из train не выполняется. В full используются все 946 фото valid, в preview — 6. Все объекты фотографии остаются вместе; исходное разбиение видно в `source_split` manifest.
- Уникальные классы MyFoodRepo добавляются в конец словаря, подтверждённые совпадения получают ID FPB. COCO category ID не используется как индекс массива.
- Пиксельные дубликаты удаляются с приоритетом val; конфликтующая разметка исключается с записью причины. Группировка имён FPB не гарантирует отсутствия всех похожих ракурсов одной порции.
- В обучение допускаются только `fpb` и `myfoodrepo`; Food-101 не создаёт рамок и не участвует в разбиениях.

Совпадение `group_id` между выборками допускается и не приводит к исключению записей. После обновления правил очистки один раз включите `OVERWRITE = True`: прежние сборки не переиспользуются, поскольку версия пайплайна изменена. Исходные файлы используются повторно.

In [ ]:
if prepared is None:
    OUTPUT, merged, summary = dp.build_dataset(
        DATA, cats, records, decisions, mode=MODE,
        seed=SEED, overwrite=OVERWRITE,
    )
else:
    OUTPUT, merged, summary = prepared
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Результат:", OUTPUT.relative_to(ROOT))
display(merged.groupby(["source", "split", "box_origin"]).size().rename("images").to_frame())


## 4. Визуальная проверка сохранённых файлов

Изображения, TXT-рамки и имена классов заново читаются из итоговой папки. Это проверяет результат записи и переназначения ID. Графики остаются только в выводе ноутбука.

In [ ]:
def show_saved(rows, title):
    config = dp.yaml.safe_load((OUTPUT / "data.yaml").read_text(encoding="utf-8"))
    visual = []
    for row in rows:
        row = dict(row)
        parsed = dp.parse_yolo((OUTPUT / row["label"]).read_text(encoding="utf-8"), len(config["names"]))
        row["objects"] = [{"class_id": obj["source_class_id"], "bbox": obj["bbox"]} for obj in parsed]
        visual.append(row)
    fig = dp.show_records(visual, config["names"], output_root=OUTPUT, count=8, title=title)
    if fig is not None:
        plt.close(fig)

labeled = merged[merged.label.notna()]
show_saved(labeled[labeled.source == "fpb"].head(8).to_dict("records"), "FPB: сохранённая разметка")
show_saved(labeled[labeled.source == "myfoodrepo"].head(8).to_dict("records"), "MyFoodRepo v2.1: рамки объектов")
show_saved(labeled[(labeled.source == "myfoodrepo") & (labeled.split == "val")].head(8).to_dict("records"),
           "MyFoodRepo v2.1: авторский valid → val")

In [ ]:
shared_gallery = []
examples = labeled.to_dict("records")
for class_id in range(len(cats["fpb"]["names"])):
    matches = {}
    for row in examples:
        if row["source"] not in matches and any(o["class_id"] == class_id for o in row["objects"]):
            matches[row["source"]] = row
    if set(matches) == {"fpb", "myfoodrepo"}:
        print("Общий ID", class_id, "—", cats["fpb"]["names"][class_id])
        shared_gallery.extend([matches["fpb"], matches["myfoodrepo"]])
    if len(shared_gallery) >= 8:
        break
show_saved(shared_gallery, "Общие классы FPB и MyFoodRepo")

### 4.1. Примеры `pixel_duplicate` и `conflicting_duplicate`

Примеры выбираются из `excluded_records.json` **текущей сборки `OUTPUT`**: до одной пары для каждой причины и каждого источника. Изображения исключённых записей и их разметка читаются из `data/raw/`, сохранённые записи — из manifest. Сравниваются SHA-256 декодированных RGB-пикселей с учётом размеров и разметка после объединения классов. Сборка, скачивание и распаковка повторно не выполняются.

- **`pixel_duplicate`**: пиксели совпадают, доступная разметка не конфликтует. Слева показана сохранённая запись, справа — исключённая копия. Если сохранена неразмеченная запись test, отсутствие рамок отмечается отдельно.
- **`conflicting_duplicate`**: пиксели совпадают, но различаются итоговые классы, количество объектов или координаты рамок. Обе показанные записи исключены; правило исключает всю конфликтующую группу. Даже небольшая разница координат может считаться конфликтом: при сравнении они округляются до восьми знаков. Порядок объектов не влияет на сравнение.

Таблицы под изображениями содержат исходные ID и координаты рамок; исходные фотографии и метки остаются на диске. В `preview` таких исключений может не быть — тогда выводится сообщение. Для просмотра примеров полной сборки выберите `MODE = "full"` и запустите ноутбук с `OVERWRITE = False`, если сборка уже готова. Сравнение выполняется по пикселям; совпадение `group_id` не исключает изображения.


In [ ]:
from collections import defaultdict

excluded_path = OUTPUT / "excluded_records.json"
duplicate_audit = dp.read_json(excluded_path) if excluded_path.exists() else []
duplicate_reasons = ("pixel_duplicate", "conflicting_duplicate")
duplicate_audit = [r for r in duplicate_audit if r["reason"] in duplicate_reasons]
duplicate_examples = []
duplicate_warnings = []
duplicate_raw_cache, duplicate_mfr_indexes = {}, {}

def duplicate_signature(row):
    if row["objects"] is None:
        return None
    return tuple(sorted((o["class_id"], tuple(round(v, 8) for v in o["bbox"])) for o in row["objects"]))

def duplicate_raw(row):
    key = (row["source"], row["source_id"])
    if key in duplicate_raw_cache:
        return duplicate_raw_cache[key]
    split, stem = row["source_id"].split("/", 1)
    if row["source"] == "fpb":
        folder = DATA / "raw/fpb" / cats["fpb"]["revision"] / split
        image = next((folder / "images" / (stem + suffix) for suffix in sorted(dp.IMAGE_SUFFIXES)
                      if (folder / "images" / (stem + suffix)).is_file()), None)
        if image is None:
            raise FileNotFoundError(folder / "images" / stem)
        objects = None if split == "test" else dp.parse_yolo(
            (folder / "labels" / (stem + ".txt")).read_text(encoding="utf-8"), len(cats["fpb"]["names"]))
    else:
        if split not in duplicate_mfr_indexes:
            duplicate_mfr_indexes[split] = {r["source_id"]: r for r in mfr.index(DATA, cats["myfoodrepo"], split)}
        original = duplicate_mfr_indexes[split][row["source_id"]]
        if split in cats["myfoodrepo"]["archives"]:
            state = dp.read_json(DATA / "metadata/myfoodrepo_v2_1_archives" / split / "extracted.json")
            folder = DATA / "raw/myfoodrepo_v2_1_archives" / split / state["image_root"]
        else:
            folder = DATA / "raw/myfoodrepo_v2_1/images"
        image = folder / Path(original["file_name"]).name
        objects = original["objects"]
    mapped = None if objects is None else [
        {"class_id": source_to_id[(row["source"], o["source_class_id"])], "bbox": o["bbox"]} for o in objects
    ]
    target_split = "fpb_test_unlabeled" if row["source"] == "fpb" and split == "test" else split
    record = dict(row, image=str(image), objects=mapped, source_split=split, split=target_split)
    record["pixel_sha256"] = dp.image_info(image)[2]
    duplicate_raw_cache[key] = record
    return record

if duplicate_audit:
    display(pd.DataFrame(duplicate_audit).groupby(["reason", "source"]).size().rename("excluded_images").to_frame())
    retained_by_hash = {r["pixel_sha256"]: r for r in merged.to_dict("records")}
    for reason in duplicate_reasons:
        pending = defaultdict(list)
        shown_sources = set()
        for excluded in (r for r in duplicate_audit if r["reason"] == reason):
            if excluded["source"] in shown_sources:
                continue
            try:
                raw = duplicate_raw(excluded)
                digest = raw["pixel_sha256"]
                if reason == "pixel_duplicate":
                    retained = retained_by_hash.get(digest)
                    if retained is None:
                        continue
                    # В сохранённом manifest уже записана итоговая разметка.
                    first = dict(retained, image=str(OUTPUT / retained["image"]))
                    signatures = [s for s in (duplicate_signature(first), duplicate_signature(raw)) if s is not None]
                    if len(set(signatures)) > 1:
                        raise ValueError("Разметка исходников отличается от завершённой сборки: проверьте OVERWRITE.")
                    pair = (first, raw)
                else:
                    first = next((r for r in pending[digest] if duplicate_signature(r) != duplicate_signature(raw)), None)
                    if first is None:
                        pending[digest].append(raw)
                        continue
                    pair = (first, raw)
                duplicate_examples.append({"reason": reason, "pixel_sha256": digest, "records": pair})
                shown_sources.add(excluded["source"])
            except FileNotFoundError as error:
                duplicate_warnings.append(str(error))
        if not any(e["reason"] == reason for e in duplicate_examples):
            print(f"{reason}: подходящие пары в этой сборке не найдены.")
else:
    print(f"В {OUTPUT.name} нет исключений pixel_duplicate и conflicting_duplicate.")
if duplicate_warnings:
    print("Не найдены исходные файлы для части примеров:")
    for missing in sorted(set(duplicate_warnings))[:5]:
        print(missing)


In [ ]:
from matplotlib.patches import Rectangle

for example in duplicate_examples:
    reason = example["reason"]
    states = ("СОХРАНЕНО", "ИСКЛЮЧЕНО") if reason == "pixel_duplicate" else ("ИСКЛЮЧЕНО · вариант A", "ИСКЛЮЧЕНО · вариант B")
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    identity_rows, object_rows = [], []
    for side, (axis, row, state) in enumerate(zip(axes, example["records"], states), start=1):
        with Image.open(row["image"]) as photo:
            axis.imshow(photo.convert("RGB"))
            width, height = photo.size
        for number, obj in enumerate(row["objects"] or [], start=1):
            x, y, w, h = obj["bbox"]
            color = "lime" if state == "СОХРАНЕНО" else "#ff5555"
            axis.add_patch(Rectangle(((x-w/2)*width, (y-h/2)*height), w*width, h*height,
                                     fill=False, edgecolor=color, linewidth=2))
            axis.text((x-w/2)*width, (y-h/2)*height, names[obj["class_id"]], fontsize=9,
                      bbox={"facecolor": color, "alpha": 0.85, "pad": 1})
            object_rows.append({"side": side, "object": number, "class": names[obj["class_id"]],
                                "class_id": obj["class_id"], "x_center": x, "y_center": y, "width": w, "height": h})
        axis.set_title(f"{state}\n{row['source']} · {row['split']} · объектов: {len(row['objects']) if row['objects'] is not None else 'разметка отсутствует'}", fontsize=10)
        axis.axis("off")
        identity_rows.append({"side": side, "state": state, "source": row["source"], "source_id": row["source_id"]})
    fig.suptitle(f"{reason} · одинаковые пиксели\nSHA-256: {example['pixel_sha256'][:20]}…", fontsize=12)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    display(pd.DataFrame(identity_rows))
    if object_rows:
        display(pd.DataFrame(object_rows).style.format({k: "{:.10f}" for k in ("x_center", "y_center", "width", "height")}))


## 5. Файлы результата

`images/`, `labels/`, `data.yaml`, списки выборок, `class_mapping.csv` и `manifest.jsonl`. `fpb_eval.yaml` использует только FPB val для отдельной оценки. `excluded_records.json` создаётся при исключении изображений. Массы и статистического `build_complete.json` нет.

Версии источников — `data/metadata/fpb_myfoodrepo_archives_sources.lock.json` (при локальном valid); прежний режим без архивов использует `fpb_myfoodrepo_sources.lock.json`; решения — `data/review/fpb_myfoodrepo_class_decisions.csv`. Подпись параметров и контрольная сумма manifest для повторного запуска — `data/metadata/fpb_myfoodrepo_<mode>.build.json`. Копии этих файлов в processed не создаются.